# 00 Preparación del entorno de trabajo

Fase 1 del plan de Ingeniería de Software II. La Fase 2 del plan corresponde
al notebook 01 de datos. Aquí se verifican entorno, Drive, rutas, seeds y una
operación CUDA diminuta si existe GPU. No se carga ningún modelo o peso ni se
entrena. Esta ejecución no valida la compatibilidad de futuras extensiones de
Vision Mamba. Puede usarse cualquier GPU compatible asignada, sin asumir A100.

## 1. Rutas y controles de ejecución

Estructura esperada en Drive:

```text
MyDrive/Proyecto_Tuberculosis/
  tbx11k_fases_1_y_2_colab.zip
  TBX11K.zip              # alternativa preferida si existe
  TBX11K/                 # o la carpeta con imgs/, lists/, annotations/
  results/                # se crea al exportar resultados
```

Ajuste `DRIVE_BASE` si usa otra ubicación. Un ZIP puede contener `TBX11K/` o
directamente `imgs/`, `lists/` y `annotations/`. No suba el ZIP del proyecto
dentro del dataset. `DATA_SOURCE_KIND='auto'` prefiere el ZIP existente.

In [ ]:
import importlib.util
import json
import os
import sys
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
DRIVE_BASE = Path('/content/drive/MyDrive/Proyecto_Tuberculosis')
PROJECT_ARCHIVE = DRIVE_BASE / 'tbx11k_fases_1_y_2_colab.zip'
DATA_SOURCE_KIND = 'auto'  # 'auto', 'zip' o 'folder'
COPY_DATASET_TO_LOCAL = True
VERIFY_ALL_IMAGE_HASHES = IN_COLAB  # recomendado True en Colab tras la primera transferencia
REBUILD_MANIFESTS_FOR_COMPARISON = False  # solo compara, no sobreescribe los congelados
RUN_COLAB_DATALOADER_CHECK = False  # un lote CPU; sin modelo ni entrenamiento
LOCAL_WORK = Path('/content/tbx11k_phase1')
# Si se revisa localmente, se usan exclusivamente datos y rutas del proyecto:
LOCAL_PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'src/tbxdata').exists() else Path.cwd().parent
RUN_NAME = 'phase1_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
print('Google Colab:', IN_COLAB)

## 2. Montar Drive y cargar el proyecto pequeño

El montaje solicita autorización de Google en Colab. La extracción valida las
rutas del ZIP y rechaza enlaces simbólicos. Puede volver a ejecutar tras una
desconexión: el código se extrae nuevamente; no se borra el dataset de Drive.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    import stat
    import zipfile
    from pathlib import PurePosixPath
    if not PROJECT_ARCHIVE.is_file():
        raise FileNotFoundError(f'Suba primero el ZIP del proyecto: {PROJECT_ARCHIVE}')
    PROJECT_ROOT = LOCAL_WORK / 'project'
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PROJECT_ARCHIVE) as archive:
        for item in archive.infolist():
            rel = PurePosixPath(item.filename)
            target = (PROJECT_ROOT / item.filename).resolve()
            if rel.is_absolute() or '..' in rel.parts or '\\' in item.filename or ':' in item.filename or PROJECT_ROOT.resolve() not in target.parents:
                raise ValueError('Ruta no segura en el ZIP del proyecto: ' + item.filename)
            if stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError('Enlace no permitido en el ZIP del proyecto.')
        archive.extractall(PROJECT_ROOT)
else:
    PROJECT_ROOT = LOCAL_PROJECT_ROOT.resolve()
    if not (PROJECT_ROOT / 'configs/data.json').is_file():
        raise FileNotFoundError('Ejecute desde la raíz del proyecto o su carpeta notebooks/.')
print('Proyecto:', PROJECT_ROOT)

## 3. Dependencias ligeras y entorno

Solo NumPy y Pillow (`requirements-data.txt`). Se conservan versiones instaladas
que cumplan los rangos y se registra el entorno exacto al finalizar. Las pruebas
locales se hicieron con Python 3.14.7, NumPy 2.5.4 y Pillow 12.3.0. Este notebook
evita reinstalar PyTorch o CUDA en Colab. Una GPU es opcional para esta fase.

In [ ]:
if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           '--disable-pip-version-check', '-r', str(PROJECT_ROOT / 'requirements-data.txt')])
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import platform
import numpy as np
import PIL
from PIL import Image
from tbxdata.dataset import TBXDataset, load_config, class_weights
from tbxdata.common import read_csv, sha256
from tbxdata.validate import validate_manifests
environment = {'python': platform.python_version(), 'numpy': np.__version__, 'pillow': PIL.__version__,
               'platform': platform.platform(), 'in_colab': IN_COLAB}
if IN_COLAB:
    try:
        import torch
    except ImportError:
        print('PyTorch no disponible: puede ejecutar toda la fase de datos y omitir el lote opcional.')
    else:
        environment['torch'] = torch.__version__
        environment['cuda_available'] = torch.cuda.is_available()
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            environment['gpu'] = {'name': props.name, 'total_memory_gib': round(props.total_memory / 2**30, 2)}
print(json.dumps(environment, indent=2))

## 4. Usar el dataset local de Colab

Para reducir lecturas pequeñas en Drive, prefiera el ZIP y extracción temporal.
Para una carpeta se usa una copia reanudable por tamaño. Un marcador reutiliza
la copia terminada; la validación posterior siempre vuelve a comprobarla.
Si la fuente cambió, use otra carpeta `LOCAL_WORK` para conservar el cache previo.
El almacenamiento `/content` es temporal y se pierde al reiniciar el entorno.

In [ ]:
from tbxdata.colab import stage_dataset
if IN_COLAB:
    zip_source = DRIVE_BASE / 'TBX11K.zip'
    folder_source = DRIVE_BASE / 'TBX11K'
    if DATA_SOURCE_KIND not in {'auto', 'zip', 'folder'}:
        raise ValueError('DATA_SOURCE_KIND debe ser auto, zip o folder.')
    source = zip_source if DATA_SOURCE_KIND == 'zip' or (DATA_SOURCE_KIND == 'auto' and zip_source.exists()) else folder_source
    DATASET_ROOT = stage_dataset(source, LOCAL_WORK / 'data_cache', COPY_DATASET_TO_LOCAL)
    RUN_DIR = LOCAL_WORK / 'runs' / RUN_NAME
else:
    if VERIFY_ALL_IMAGE_HASHES:
        raise RuntimeError('La revisión local reutiliza la auditoría; deje VERIFY_ALL_IMAGE_HASHES=False.')
    DATASET_ROOT = PROJECT_ROOT / 'TBX11K'
    RUN_DIR = PROJECT_ROOT / 'reports' / 'notebook_runs' / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET_ROOT)
print('Evidencias de esta ejecución:', RUN_DIR)

## Seeds y comprobación del dispositivo

Se fijan seeds para random, NumPy y PyTorch, si se ejecuta en Colab. El uso de
algoritmos deterministas se registra, sin prometer equivalencia absoluta entre
hardware o versiones. Para la GPU: Entorno de ejecución, Cambiar tipo de entorno
de ejecución, seleccionar GPU. Si no hay GPU, se conserva la evidencia CPU y
el resultado GPU pendiente. No reinstale CUDA/PyTorch a ciegas.

In [ ]:
from tbxdata.environment import inspect_environment
from tbxdata.dataset import load_config
config = load_config(PROJECT_ROOT / 'configs/data.json')
setup_report = inspect_environment(config['seed'], verify_cuda=IN_COLAB)
print(json.dumps(setup_report, indent=2))

## Comprobar la transferencia completa

Se reutilizan los hashes existentes. En Colab se comparan los 12.278 archivos
para cerrar la comprobación pendiente del primer envío. No se explora de nuevo
el dataset ni se modifican particiones. Localmente solo se usan muestras.

In [ ]:
transfer_report = validate_manifests(PROJECT_ROOT, DATASET_ROOT,
                                    hash_samples_per_class=2,
                                    all_image_hashes=IN_COLAB)
print('Integridad:', transfer_report['status'])
print('Archivos con SHA256 verificado completo:', transfer_report['all_image_hashes_verified'])

## Experimentos y checkpoints futuros

Las carpetas de salida se preparan en Drive y los resultados se guardan al
finalizar. No se generan checkpoints de modelos en estas fases. Los pesos
futuros deben permanecer fuera del repositorio Git.

In [ ]:
import shutil
from datetime import datetime, timezone
setup_name = 'entorno_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
setup_output = (DRIVE_BASE / 'results' / setup_name) if IN_COLAB else (PROJECT_ROOT / 'reports' / 'environment' / setup_name)
setup_output.mkdir(parents=True, exist_ok=True)
if IN_COLAB:
    EXPERIMENTS_ROOT = DRIVE_BASE / 'experiments'
    CHECKPOINTS_ROOT = DRIVE_BASE / 'checkpoints'
    EXPERIMENTS_ROOT.mkdir(parents=True, exist_ok=True)
    CHECKPOINTS_ROOT.mkdir(parents=True, exist_ok=True)
    setup_report['output_paths'] = {'experiments': str(EXPERIMENTS_ROOT), 'checkpoints': str(CHECKPOINTS_ROOT)}
(setup_output / 'environment_setup.json').write_text(json.dumps(setup_report, indent=2), encoding='utf-8')
(setup_output / 'transfer_integrity.json').write_text(json.dumps(transfer_report, indent=2), encoding='utf-8')
shutil.copy2(PROJECT_ROOT / 'configs/data.json', setup_output / 'config.json')
shutil.copy2(PROJECT_ROOT / 'manifests/manifest_metadata.json', setup_output / 'manifest_metadata.json')
print('Entregar esta carpeta para actualizar el informe:', setup_output)
if setup_report['cuda_probe']['status'] != 'passed':
    print('GPU/CUDA no verificados en esta ejecución; no presentar como completados.')

## Evidencias para cerrar las dos fases del curso

Entregar este notebook ejecutado y la carpeta entorno creada en Drive. Para
declarar GPU verificada, environment_setup.json debe mostrar cuda_probe passed.
transfer_integrity.json debe registrar 12.278 hashes completos. El notebook 01
ya tiene evidencia de datos en CPU; no necesita repetir el entrenamiento porque
no existe entrenamiento en estas dos fases. La URL y commit de GitHub se
documentarán al publicar el repositorio del equipo. No se promete una A100.